# Libs

In [ ]:
import sys
sys.path.append("../libs/")
sys.path.append("../")

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from scipy.signal import find_peaks
from scipy.integrate import trapezoid

warnings.filterwarnings('ignore')

DIR_DATA = os.getcwd()+"/data/"
DIR_OUTPUT = os.getcwd()+"/output/"

# Load Data

In [ ]:
base_name = "Crystallizer #2.csv"

df_dataset = pd.read_csv(DIR_DATA + base_name, sep=";", decimal=".")
df_dataset["TIMESTAMP"] = pd.to_datetime(df_dataset["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_dataset["Resultado de Ferro (ppm)"] = pd.to_numeric(df_dataset["Resultado de Ferro (ppm)"], errors="coerce")
df_dataset.sort_values(by="TIMESTAMP", inplace=True)
df_dataset

In [ ]:
df_duplicados = df_dataset[df_dataset.duplicated(subset=['Labref'], keep=False)]
df_duplicados

In [ ]:
# Aplicando média para medidas com Labref iguais
agg_logic = {col: 'mean' if df_dataset[col].dtype.kind in 'biufc' else 'first' 
             for col in df_dataset.columns if col != 'Labref'}
df_dataset = df_dataset.groupby('Labref', as_index=False).agg(agg_logic)
df_dataset

In [ ]:
idx = df_dataset[df_dataset['Labref'] == 4027521].index # Removendo amostra 4027521 que possui valor muito discrepante
df_dataset = df_dataset.drop(idx)
df_dataset

# Load Events

In [ ]:
base_name_eventos = "Eventos-Reator2_datasOriginais.csv"

df_eventos = pd.read_csv(DIR_DATA + base_name_eventos, sep=";", decimal=".")
df_eventos["TIMESTAMP"] = pd.to_datetime(df_eventos["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_eventos

In [ ]:
df_eventos.drop([0,1,2], inplace=True)
df_eventos

# Plot data

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_dataset['TIMESTAMP'],
    y=df_dataset["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos.iterrows():
    fig.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",  # y vai de 0 a 1 cobrindo todo o gráfico
        line=dict(color="red", width=1.5, dash="dash")
    )
    # Anotação separada (opcional)
    fig.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig.update_layout(
    template='plotly_white',
    hovermode='x unified'
)
fig.show()

In [ ]:
fig.write_html("Grafico2.html")

# Clustering events

In [ ]:
def extract_features(df_window):
    """Extrai características da janela para detecção de anomalias."""
    if len(df_window) < 2:
        return None # Ignora janelas com menos de 2 pontos
    
    # Tempo em horas (relativo ao início da janela) para cálculos matemáticos
    t_hours = (df_window['TIMESTAMP'] - df_window['TIMESTAMP'].min()).dt.total_seconds() / 3600.0
    y_ppm = df_window['Resultado de Ferro (ppm)'].values
    
    # Derivadas (Taxa de variação ppm/hora)
    dt = np.diff(t_hours)
    dy = np.diff(y_ppm)
    # Evita divisão por zero se houver amostras no exato mesmo segundo
    rates = np.divide(dy, dt, out=np.zeros_like(dy), where=dt!=0) 
    
    # Inclinação Linear (Slope)
    lr = LinearRegression().fit(t_hours.values.reshape(-1, 1), y_ppm)
    slope = lr.coef_[0]
    
    # Integral (Área sob a curva)
    area = trapezoid(y=y_ppm, x=t_hours)
    
    # Variância do delta T (em horas)
    dt_var = np.var(dt) if len(dt) > 1 else 0
    
    # Foco no Tempo Recente (Último movimento antes do alarme)
    last_dy = dy[-1] if len(dy) > 0 else 0
    last_dt = dt[-1] if len(dt) > 0 else 0
    ema_final = df_window['Resultado de Ferro (ppm)'].ewm(span=len(df_window), adjust=False).mean().iloc[-1]
    
    # Complexidade e Previsibilidade (Ruído vs Tendência)
    # np.sign retorna direção (-1, 0, 1). np.diff != 0 conta quantas vezes a direção mudou.
    inversoes_tendencia = np.sum(np.diff(np.sign(rates)) != 0) if len(rates) > 1 else 0
    
    # Análise de Picos e Valores Extremos
    rms = np.sqrt(np.mean(y_ppm**2))
    crest_factor = np.max(np.abs(y_ppm)) / rms if rms > 0 else 0
    
    # Encontra picos e extrai a maior proeminência (destaque do pico em relação à base)
    picos, propriedades = find_peaks(y_ppm, prominence=0)
    max_prominence = np.max(propriedades['prominences']) if len(picos) > 0 else 0
    
    return {
        'taxa_max': np.max(rates) if len(rates) > 0 else 0,
        'taxa_media': np.mean(rates) if len(rates) > 0 else 0,
        'slope': slope,
        'area_curva': area,
        'dt_variancia': dt_var,
        'ppm_max': np.max(y_ppm),
        'ppm_min': np.min(y_ppm),
        'ppm_media': np.mean(y_ppm),
        'ppm_std': np.std(y_ppm),
        'ultimo_dy': last_dy,
        'ultimo_dt': last_dt,
        'ema_final': ema_final,
        'inversoes_tendencia': inversoes_tendencia,
        'crest_factor': crest_factor,
        'max_prominence': max_prominence
    }


DIAS_JANELA = 15

features_list = []
valid_events = []

eventos_timestamps = df_eventos["TIMESTAMP"]

# Extração de Features por Janela
for evento in eventos_timestamps:
    inicio_janela = evento - pd.Timedelta(days=DIAS_JANELA)
    
    # Filtra os dados apenas para a janela ANTES do evento
    mask = (df_dataset['TIMESTAMP'] >= inicio_janela) & (df_dataset['TIMESTAMP'] < evento)
    df_window = df_dataset[mask]
    
    feats = extract_features(df_window)
    if feats:
        features_list.append(feats)
        valid_events.append(evento)

# Criação do DataFrame de Features
df_features = pd.DataFrame(features_list, index=valid_events)
df_features

In [ ]:
# Padronização e Clusterização
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features)

In [ ]:
# Elbow Method
inercia = []
K_range = range(1, len(df_eventos)-1) # Testa de 1 a 10 clusters

for k in K_range:
    kmeans_teste = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_teste.fit(X_scaled)
    inercia.append(kmeans_teste.inertia_)

# Plota o gráfico para visualização
plt.figure(figsize=(8, 5))
plt.plot(K_range, inercia, marker='o', linestyle='--')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia')
plt.title('Método do Cotovelo para K Ideal')
plt.xticks(K_range)
plt.grid(True)
plt.show()

In [ ]:
# Clusterização
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_features['Cluster'] = kmeans.fit_predict(X_scaled)
# Resultado final
df_features

In [ ]:
# Dicionário de cores para os clusters
cores_clusters = {
    0: 'blue',   
    1: 'orange', 
    2: 'green'   
}

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_dataset['TIMESTAMP'],
    y=df_dataset["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos.iterrows():
    evento_ts = row["TIMESTAMP"]
    
    # Adiciona o sombreado da janela baseado no Cluster
    if evento_ts in df_features.index:
        cluster = df_features.loc[evento_ts, 'Cluster']
        cor_fundo = cores_clusters.get(cluster, 'gray')
        inicio_janela = evento_ts - pd.Timedelta(days=DIAS_JANELA)
        
        fig.add_vrect(
            x0=inicio_janela,
            x1=evento_ts,
            fillcolor=cor_fundo,
            opacity=0.2, 
            layer="below", 
            line_width=0,
            annotation_text=f"C{cluster}",
            annotation_position="top left"
        )

    # Linha vertical exata do evento
    fig.add_shape(
        type="line",
        x0=str(evento_ts),
        x1=str(evento_ts),
        y0=0,
        y1=1,
        yref="paper", 
        line=dict(color="red", width=1.5, dash="dash")
    )
    
    # Anotação
    fig.add_annotation(
        x=str(evento_ts),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title='Série Temporal com Sombreado dos Clusters (Janela de Análise)'
)
fig.show()